In [ ]:
import pandas as pd

df = pd.read_csv("emails.csv")

# 1. Check data file
print(df.head()) #xem vài dòng đầu của dataset
print(df.shape)
print(df.columns)
print(df.info())

                                                text  spam
0  Subject: naturally irresistible your corporate...     1
1  Subject: the stock trading gunslinger  fanny i...     1
2  Subject: unbelievable new homes made easy  im ...     1
3  Subject: 4 color printing special  request add...     1
4  Subject: do not have money , get software cds ...     1
(5728, 2)
Index(['text', 'spam'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5728 non-null   str  
 1   spam    5728 non-null   int64
dtypes: int64(1), str(1)
memory usage: 89.6 KB
None


In [ ]:
# 2. Check missing data
print(df.isnull().sum())
# df = df.dropna() #dùng để xoá dòng bị thiếu data

text    0
spam    0
dtype: int64


In [ ]:
# 3. Check label có cân bằng không (vì nếu quá lệch nhau => accuracy bị ảnh hưởng)
# Độ lệch 70:30 đến 85:15 là đẹp
print(df["spam"].value_counts())

spam
0    4360
1    1368
Name: count, dtype: int64


In [7]:
# 4. Tách input & output
# input X: email content
X = df["text"]    
# output y: label
y = df["spam"]

In [15]:
# 5. Text cleaning
import re

def clean(text):
    # 1. Chuyển toàn bộ thành chữ thường
    text = text.lower()
    # 2. Xoá chữ 'Subject' ở đầu mỗi text
    text = re.sub(r'^\s*subject\s*:?[\s]*', '', text)
    # 3. Xoá URL
    text = re.sub(r'http\S+|www\S+',' ',text)
    # 4. Xóa HTML tags
    text = re.sub(r'<.*?>',' ',text)
    # 5. Xoá mọi thứ KHÔNG phải chữ cái
    text = re.sub(r'[^a-zA-Z ]',' ',text)
    # 6. Dọn khoảng trắng thừa
    text = re.sub(r'\s+',' ',text).strip()

    return text

df["text"] = df["text"].apply(clean)

df.to_csv(
    "emails_cleaned.csv",
    index=False
)

In [16]:
# 6. Chuyển text → số (Vì ML ko hiểu chữ)
from sklearn.feature_extraction.text import TfidfVectorizer

# X = input features (text)
X = df["text"]

# y = label
y = df["spam"]

# Create TF-IDF converter
vectorizer = TfidfVectorizer()

# Convert text -> numbers
X = vectorizer.fit_transform(X)

print(X.shape)

(5728, 33684)


In [ ]:
# Chia train / test
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)